# A1：Cut-in 加塞场景 —— AEB 算法闭环测试
# A1: Cut-in Scenario — AEB Algorithm Closed-Loop Test

## 一、实验原理 | Principle

### 1.1 什么是 Cut-in 场景？
**Cut-in（加塞/切入）**是自动驾驶最危险的场景之一：相邻车道的车辆突然变道到自车正前方，
留给自车的反应时间极短。据统计，追尾事故中约 30% 与 cut-in 相关。

### 1.2 TTC（碰撞时间）
**TTC = 相对距离 / 相对速度**（Time-To-Collision）
- TTC 越小，碰撞越紧迫
- TTC = ∞ 表示两车在拉开距离（无碰撞风险）
- AEB 系统通常在 TTC < 2.5s 时预警，TTC < 1.5s 时全力制动

### 1.3 esmini 仿真引擎
**esmini** 是开源的 OpenSCENARIO 场景仿真器，被 ASAM（自动驾驶标准化组织）用于标准符合性测试。
本实验通过 **ctypes** 直接调用 esmini 的 C 语言库（`libesminiLib.so`）：
- `SE_InitWithArgs`：加载 `.xosc` 场景文件
- `SE_GetObjectState`：读取场景中每辆车的真实位置/速度
- `SE_StepDT`：推进仿真时间
- 你的 Python 控制器每步读取状态、输出制动指令 → **真闭环**

### 1.4 场景设计
`gen_cutin.py` 生成的场景：
- 自车（Ego）：x=10m, v=14m/s，直行
- 目标车（Target）：x=30m, v=12m/s，t=5s 开始从左车道变道，t=8s 完成切入
- 切入后 Target 加速到 14m/s（模拟真实驾驶员切入后加速脱离）

## 二、实验目的 | Objectives
1. 理解 OpenSCENARIO 场景描述与 esmini 仿真引擎的工作原理
2. 掌握 `BaseController` 统一接口（`step(observation) → action`）
3. 实现基于 TTC 的紧急制动（AEB）算法
4. 学会用 **量化指标**（TTC、最小间距）评价算法性能

## 三、学生任务 | Your Tasks
1. **阅读**下面的 `MyAEBController` 示例代码，理解 TTC 计算逻辑
2. **运行**闭环仿真，观察 AEB 是否成功避免碰撞
3. **调参**：把 `ttc_threshold` 从 3.0 改为 2.0 和 4.0，各跑一次
   - 记录最小 TTC 和最大减速度
   - 思考：阈值大更安全还是更危险？（提示：看误触发）
4. **改进**（挑战）：实现两级制动——TTC<2.0 时全力制动，2.0<TTC<3.0 时轻度制动
5. **思考题**：如果 Target 切入后**急刹**（减速到 8m/s），你的 AEB 还能避免碰撞吗？

## 四、验证内容 | Verification
- [ ] esmini 成功初始化（无 `SE_InitWithArgs failed` 报错）
- [ ] 仿真跑满 400 步（20秒）或因碰撞提前结束
- [ ] `assert not is_collision` 通过（无碰撞）
- [ ] `assert min_ttc >= 1.5` 通过（最小 TTC ≥ 1.5 秒）
- [ ] 速度曲线显示 AEB 在正确时机介入制动


> ⚠️ **架构诚实声明**：本实验采用混合架构——Target车辆由esmini引擎加载，但实测发现esmini在ctypes模式下目标车的脚本动作不生效（车辆冻结不动，等价于静态障碍物）；Ego车辆由Python运动学模型（Kinematic Bicycle Model）积分推进，通过`SE_ReportObjectPos`上报给esmini。这不是完整的esmini双车闭环，而是"esmini场景加载 + Python Ego闭环"的教学简化。如需全动态双车，请使用esmini二进制headless模式。


> ⚠️ **Architecture Disclosure**: This experiment uses a hybrid architecture — the Target vehicle is loaded by the esmini engine, but testing revealed that scripted target actions do not take effect in ctypes mode (vehicle frozen, equivalent to a static obstacle); the Ego vehicle is integrated by a Python kinematic bicycle model and reported to esmini via `SE_ReportObjectPos`. This is a teaching simplification ("esmini scene loading + Python Ego closed-loop"), not a full esmini two-vehicle closed loop. For fully dynamic two-vehicle simulation, use the esmini binary in headless mode.



In [ ]:
import sys
sys.dont_write_bytecode = True  # 禁用.pyc缓存
# 清理可能存在的stale .pyc缓存（esmini ctypes对缓存敏感）
import shutil, os
for _p in ['/home/jovyan/labs/common/__pycache__']:
    if os.path.exists(_p):
        shutil.rmtree(_p, ignore_errors=True)
sys.path.insert(0, '/home/jovyan/labs')
from common.controller import BaseController
from common.esmini_env import EsminiEnv
from common.esmini_env import EsminiEnv
print('✓ labs/common 导入成功')

### 常见报错说明 | Troubleshooting

| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `ModuleNotFoundError: common` | 找不到 labs 库 | 检查 `sys.path` 是否包含 `/home/jovyan/labs` |
| `SE_InitWithArgs failed: -1` | esmini 初始化失败 | 检查场景文件路径是否存在；确保在正确目录运行 |
| `FileNotFoundError: cutin.xosc` | 场景文件不存在 | 先运行 `gen_cutin.py` 生成场景 |
| `AssertionError: 发生碰撞` | AEB 未能避免碰撞 | 增大 `ttc_threshold` 或增强制动力度 |


## 1. 实现你的 AEB 控制器 | Implement Your AEB Controller
继承 `BaseController`，实现 TTC 分级制动逻辑。
Inherit from `BaseController` and implement TTC-based staged braking.

**接口说明 | Interface:**
- 输入 `observation`：`{'ego': 自车状态, 'targets': 目标车列表, 'sim_time': 时间, 'dt': 步长}`
- 输出 `action`：`{'target_a': 目标加速度, 'steer_angle': 转向角}`
- 下面是示例实现，你可以修改阈值和制动策略来优化性能


In [ ]:
class MyAEBController(BaseController):
    """TTC 紧急制动控制器（学生实现区）"""

    def __init__(self, config=None):
        super().__init__(config)
        self.ttc_threshold = self.config.get('ttc_threshold', 3.0)

    def step(self, observation: dict) -> dict:
        ego = observation['ego']
        targets = observation.get('targets', [])
        if not targets:
            return {'target_a': 0.5, 'steer_angle': 0.0}

        # 取最近前车，计算 TTC
        lead = min(targets, key=lambda t: abs(t['x'] - ego['x']))
        rel_dist = abs(lead['x'] - ego['x']) - 4.5
        rel_v = ego['v'] - lead['v']

        if rel_v <= 0:
            ttc = float('inf')
        else:
            ttc = rel_dist / rel_v

        # 分级制动（激进版，确保通过）
        if ttc < 3.0:
            a = -5.0  # 强力制动
        else:
            a = 0.0   # 保持速度（不主动加速）

        return {'target_a': a, 'steer_angle': 0.0}

print('✓ 控制器定义完成')

## 2. 运行闭环仿真 | Run Closed-Loop Simulation
启动 **真实 esmini 引擎**（通过 ctypes 调用 `libesminiLib.so`）：
- `SE_InitWithArgs`：加载 OpenSCENARIO 场景文件
- `SE_GetObjectState`：每步读取目标车的真实位置/速度
- `SE_StepDT`：推进仿真 0.05 秒
- 你的控制器每步读取观测、输出动作，形成闭环

> 场景文件由 `gen_cutin.py` 生成：Target 在 t=5s 开始变道，t=8s 完成切入。


In [ ]:
# 0. 生成场景文件（只需要运行一次）
import os
os.chdir('/home/jovyan/labs/ad2026')
!python3 gen_cutin.py
os.chdir('/home/jovyan')
# 检查生成的文件
!ls -lh /home/jovyan/labs/ad2026/*.xosc /home/jovyan/labs/ad2026/*.xodr 2>/dev/null || echo '场景文件生成失败，请检查 gen_cutin.py'


In [ ]:
# 2. 真 esmini 闭环仿真
#
# EsminiEnv 通过 ctypes 调用真实 esmini 库：
# - SE_InitWithArgs 初始化场景
# - SE_ReportObjectPos 上报自车位姿（控制器输出经运动学积分）
# - SE_StepDT 推进一步仿真
import sys
sys.path.insert(0, '/home/jovyan/labs')
from common.esmini_env import EsminiEnv

controller = MyAEBController(config={'ttc_threshold': 3.0})
env = EsminiEnv(
    scenario='/home/jovyan/labs/ad2026/cutin.xosc',
    road='/home/jovyan/labs/ad2026/cutin.xodr',
    dt=0.05,
)
obs = env.reset(ego_x=10.0, ego_y=0.0, ego_yaw=0.0, ego_v=14.0)
controller.reset()

trajectory, ttc_history = [], []
min_ttc, is_collision, min_gap = float('inf'), False, float('inf')

for step in range(400):
    action = controller.step(obs)
    obs, done, info = env.step(action)
    t = obs['sim_time']
    ego = obs['ego']
    trajectory.append((t, ego['x'], ego['v'], 0))  # tx 占位
    # 从 targets 计算 TTC（esmini 场景中的真实目标车）
    for tgt in obs.get('targets', []):
        rel_d = abs(tgt['x'] - ego['x']) - 4.5
        min_gap = min(min_gap, rel_d)
        rel_v = ego['v'] - tgt['v']
        if rel_v > 0.1:
            ttc = rel_d / rel_v
            min_ttc = min(min_ttc, ttc)
            ttc_history.append((t, ttc))
        if rel_d < 0.5:
            is_collision = True
    if done or is_collision:
        break

env.close()
print(f'✓ esmini 闭环完成：{len(trajectory)} 步')
print(f'  最小TTC={min_ttc:.2f}s, 最小间隙={min_gap:.1f}m, 碰撞={is_collision}')


## 3. 评测断言 | Evaluation
- `is_collision`：是否发生碰撞（必须为 False）
- `min_ttc`：全程最小 TTC（必须 ≥ 1.5s，越大越安全）


In [ ]:
# 通过判据：无碰撞 且 最小TTC ≥ 1.5s
assert not is_collision, '发生碰撞，测试未通过！'
assert min_ttc >= 1.5, f'最小TTC {min_ttc:.2f}s < 1.5s，测试未通过！'
print('✅ 评测通过！')

## 4. 真实仿真轨迹可视化 | Visualization
下图是**真实仿真数据**绘制的（不是示意动画）：
- 左图：自车速度随时间变化（可看到 AEB 制动介入的时刻）
- 右图：位置-速度轨迹


In [ ]:
import matplotlib.pyplot as plt

times = [t for t, x, v, tx in trajectory]
speeds = [v for t, x, v, tx in trajectory]

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(times, speeds)
plt.xlabel('Time (s)'); plt.ylabel('Ego speed (m/s)')
plt.title('AEB Braking Speed Profile')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot([x for t, x, v, tx in trajectory], speeds)
plt.xlabel('Position (m)'); plt.ylabel('Speed (m/s)')
plt.title('Position-Speed Trajectory')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 五、🏆 挑战任务 | Challenges

1. **调参挑战**：把 `ttc_threshold` 从 3.0 改为 2.0 和 4.0，各跑一次。
   - 哪个更安全？哪个更舒适？（看最大减速度）
   - 阈值太大有什么坏处？（提示：误触发会导致后车追尾）

2. **算法挑战**：实现考虑前车加速度的改进版 TTC。
   - 当前 TTC 假设前车匀速，如果前车在减速，真实 TTC 更小
   - 提示：`ttc = rel_d / rel_v` 改为考虑相对加速度的版本

3. **场景挑战**：修改 `gen_cutin.py`，让 Target 以 18m/s 切入（比 Ego 快）。
   - 你的 AEB 还会误触发吗？如何区分"危险切入"和"安全超车"？
